# Notebook 01: Tracking Pipeline

RF-DETR detection, optional watershed splitting of merged-fly bboxes, OC-SORT tracking, vial assignment, diagnostics, overlay video.

## Stages
1. Definitions
2. (Optional) Background subtraction
3. Vial ROIs
4. RF-DETR + watershed + OC-SORT
5. Vial assignment and ordered IDs
6. Diagnostics
7. Overlay videos

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import os
import shutil
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import Video

from src.preprocessing import preprocess_bgsub_gui
from src.metrics import run_diagnostics
from src.tracking import export_tracks_xy_tuple_csv_one_config
from src.stitching import wide_to_long
from src.roi import (
    draw_and_save_vial_rois,
    assign_vials_and_ordered_ids,
    load_vial_rois,
    resolve_vial_expected_counts,
)
from src.visualization import (
    render_vial_overlay_video,
    render_raw_overlay_video,
    render_detections_video,
)
from utils import (
    load_config,
    make_run_output_dir,
    save_config_snapshot,
    save_run_params,
)

## 1. Definitions

Every path and value the notebook uses is derived in the next cell from `config.yaml`. Later cells only *use* those names. To change a run, edit `config.yaml` — not the notebook. Credentials and model id come from `creds_config.yaml`; everything else comes from `config.yaml` via attribute access (`config.tracker.confidence`, etc.).

In [ ]:
# ── 1. Definitions ────────────────────────────────────────────────────
# Everything here is DERIVED from config.yaml. You do NOT edit this cell to
# change a run — you edit config.yaml. Later cells only *use* these names.

# The notebook lives in notebooks/, one level below the repo root. Every
# repo-relative path is built from REPO_ROOT so this is the only place that
# knows the notebook's location.
REPO_ROOT   = Path("..")
CONFIG_PATH = REPO_ROOT / "config.yaml"

config = load_config(CONFIG_PATH)
creds  = load_config(REPO_ROOT / "creds_config.yaml")
API_KEY, MODEL_ID = creds.API_KEY, creds.MODEL_ID

# --- Input video (chosen by the user in config.yaml -> video.raw_path) ---
RAW_VIDEO = REPO_ROOT / config.video.raw_path
assert RAW_VIDEO.exists(), f"Video not found: {RAW_VIDEO}"

# --- Run output folder + labels ---
OUTPUT_PATH = make_run_output_dir(RAW_VIDEO, outputs_root=REPO_ROOT / "data" / "outputs")
short_name  = Path(OUTPUT_PATH).name.split("_", 2)[-1]  # human label, e.g. "31DPE_n001"
stem        = RAW_VIDEO.stem

# --- Output paths (all derived from OUTPUT_PATH) ---
pp_out          = os.path.join(OUTPUT_PATH, f"{stem}_pp.mp4")
raw_cropped_out = os.path.join(OUTPUT_PATH, f"{stem}_raw_cropped.mp4")
ROI_JSON        = os.path.join(OUTPUT_PATH, "vial_rois.json")
OCSORT_CSV      = os.path.join(OUTPUT_PATH, "ocsort_tracks.csv")
DET_LOG_CSV     = os.path.join(OUTPUT_PATH, "detections_raw.csv")
LONG_CSV        = os.path.join(OUTPUT_PATH, "ocsort_tracks_long.csv")
ORDERED_CSV     = os.path.join(OUTPUT_PATH, "ordered_tracks.csv")
RAW_OVERLAY_MP4 = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_raw_ocsort.mp4")
OVERLAY_MP4     = os.path.join(OUTPUT_PATH, f"{short_name}_overlay_ordered.mp4")

# --- ROI cache (shared by preprocessing + vial cells) ---
ROI_LIBRARY = REPO_ROOT / "roi_library.json"
_video_key  = stem
_library    = json.loads(ROI_LIBRARY.read_text()) if ROI_LIBRARY.exists() else {}

# --- Behavior toggles (all from config.yaml; the notebook only reads them) ---
preprocess     = config.preprocessing.enabled
write_overlays = config.visualization.enabled
CACHED_DETS    = config.tracker.cached_detections   # repo-root-relative path, or null
_det_source = (
    str(REPO_ROOT / CACHED_DETS)
    if CACHED_DETS and (REPO_ROOT / CACHED_DETS).exists()
    else DET_LOG_CSV
)

# --- Video metadata (probed once) ---
_cap = cv2.VideoCapture(str(RAW_VIDEO))
video_info = {
    "path":   str(RAW_VIDEO),
    "fps":    _cap.get(cv2.CAP_PROP_FPS) or config.video.fallback_fps,
    "width":  int(_cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
    "height": int(_cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
    "frames": int(_cap.get(cv2.CAP_PROP_FRAME_COUNT)),
}
_cap.release()
fps = video_info["fps"]

# --- Mutable run state (reassigned by later cells) ---
PATH_TO_VID       = str(RAW_VIDEO)   # -> pp_out after preprocessing (cell 5)
RAW_CROPPED_VIDEO = None             # -> set by preprocessing (cell 5)
_crop_params      = None             # -> set by preprocessing (cell 5)
_vials            = None             # -> set by the vial-ROI cell (cell 7)

In [ ]:
# Setup: keep the raw video beside its outputs, and record run metadata.
# Hardlink to save disk; fall back to a copy if the filesystem refuses.
dest = Path(OUTPUT_PATH) / RAW_VIDEO.name
if not dest.exists():
    try:
        os.link(RAW_VIDEO, dest)
    except OSError:
        shutil.copy2(RAW_VIDEO, dest)

save_config_snapshot(OUTPUT_PATH, config_path=CONFIG_PATH)
save_run_params(OUTPUT_PATH, "video", video_info)

## 2. (Optional) Background subtraction and temporal trim

GUI: draw a crop rectangle, pick a `[start, end)` frame range. The `_pp.mp4` output is spatially cropped, temporally trimmed, and background-subtracted (85th-percentile temporal median). All downstream stages run on this clip. Crop params are cached in `roi_library.json` so re-runs skip the GUI.

In [ ]:
# `preprocess` comes from config.preprocessing.enabled (set in the Definitions cell).
# Set it to false in config.yaml to reuse a prior _pp.mp4 instead of running the GUI.
if preprocess:
    _stored_crop = _library.get(_video_key, {}).get("preprocessing") if config.roi.use_saved_roi else None

    pp_path, _crop_params = preprocess_bgsub_gui(
        video_path=str(RAW_VIDEO),
        out_mp4=pp_out,
        out_raw_mp4=raw_cropped_out,
        gain=config.preprocessing.bg_gain,
        white_level=config.preprocessing.bg_white_level,
        bg_sample_stride=config.preprocessing.bg_sample_stride,
        bg_percentile=config.preprocessing.bg_percentile,
        crop_params=_stored_crop,
    )
    PATH_TO_VID = str(pp_path)
    RAW_CROPPED_VIDEO = Path(raw_cropped_out)

    _library.setdefault(_video_key, {})["preprocessing"] = _crop_params
    _library[_video_key]["video_path"] = str(RAW_VIDEO)
    ROI_LIBRARY.write_text(json.dumps(_library, indent=2))

    with open(os.path.join(OUTPUT_PATH, "crop_roi.json"), "w") as f:
        json.dump(_crop_params, f, indent=2)
else:
    # Skip preprocessing — PATH_TO_VID stays the raw video (set in Definitions),
    # so the tracker runs on the uncropped, un-subtracted clip.
    print(f"Preprocessing skipped. Using {RAW_VIDEO.name} as tracker input.")

save_run_params(OUTPUT_PATH, "preprocessing", {
    "video_pp": str(PATH_TO_VID),
    "video_raw_cropped": str(RAW_CROPPED_VIDEO) if RAW_CROPPED_VIDEO else None,
    "crop_params": _crop_params,
})

## 3. Vial ROIs

Drag a rectangle around each vial; press **Enter** when done (**U** undo, **R** reset, **Esc** cancel). Cached in `roi_library.json` keyed by video stem so re-runs skip the GUI when `config.roi.use_saved_roi` is true.

In [ ]:
if _vials is not None:
    # Vial ROIs already provided (e.g. reused from a prior run) — write them out, skip GUI.
    with open(ROI_JSON, "w") as f:
        json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
else:
    _stored_vials = _library.get(_video_key, {}).get("vial_rois")
    if config.roi.use_saved_roi and _stored_vials is not None:
        _vials = {k: tuple(v) for k, v in _stored_vials.items()}
        with open(ROI_JSON, "w") as f:
            json.dump({k: list(v) for k, v in _vials.items()}, f, indent=2)
    else:
        _vials = draw_and_save_vial_rois(video_path=str(RAW_VIDEO), roi_json_path=ROI_JSON)
        _library.setdefault(_video_key, {})["vial_rois"] = {k: list(v) for k, v in _vials.items()}
        ROI_LIBRARY.write_text(json.dumps(_library, indent=2))

save_run_params(OUTPUT_PATH, "roi", {k: list(v) for k, v in _vials.items()})

## 4. RF-DETR, watershed split, OC-SORT

RF-DETR detection (cached to `detections_raw.csv` on first run; reused on re-runs). Watershed splits oversized bboxes that contain multiple touching flies before OC-SORT sees them; controlled by `config.watershed`. Set `tracker.cached_detections` in `config.yaml` to a prior `detections_raw.csv` to skip inference entirely.

In [ ]:
tracker_cfg = config.tracker
ghost_cfg   = config.tracker.ghost_detection

# Per-vial expected counts gate ghost detection (see src/tracking.py): the ROI
# JSON's real per-vial n_flies where present, else the config fallback per vial.
_, n_flies_dict = load_vial_rois(ROI_JSON)
vial_expected_counts = resolve_vial_expected_counts(
    n_flies_dict, _vials, config.pipeline.expected_per_vial
)

df_wide, tracker = export_tracks_xy_tuple_csv_one_config(
    video_path=str(PATH_TO_VID),
    output_csv=OCSORT_CSV,
    api_key=API_KEY,
    model_id=MODEL_ID,
    inference_api_url=config.roboflow.inference_api_url,
    detection_confidence_rfdetr=tracker_cfg.detection_confidence_rfdetr,
    confidence=tracker_cfg.confidence,
    lost_track_buffer=tracker_cfg.lost_track_buffer,
    minimum_matching_threshold=tracker_cfg.minimum_matching_threshold,
    minimum_consecutive_frames=tracker_cfg.minimum_consecutive_frames,
    asso_func=tracker_cfg.asso_func,
    brownian_pos_noise=tracker_cfg.brownian_pos_noise,
    aspect_weight=tracker_cfg.aspect_weight,
    behavioral_weights=dict(tracker_cfg.behavioral_weights),
    overlap_weight_scale=tracker_cfg.overlap_weight_scale,
    inertia=tracker_cfg.inertia,
    delta_t=tracker_cfg.delta_t,
    overlap_iou_scale=tracker_cfg.overlap_iou_scale,
    edge_fraction=tracker_cfg.edge_fraction,
    expected_count=tracker_cfg.expected_count,
    w_under=tracker_cfg.w_under,
    w_over=tracker_cfg.w_over,
    jump_factor=tracker_cfg.jump_factor,
    jump_iou_threshold=tracker_cfg.jump_iou_threshold,
    jump_inertia=tracker_cfg.jump_inertia,
    ghost_detection_enabled=ghost_cfg.enabled,
    ghost_offset_fraction=ghost_cfg.offset_fraction,
    ghost_confidence=ghost_cfg.confidence,
    ghost_occlusion_max_gap=ghost_cfg.occlusion_max_gap,
    ghost_top_exit_px=ghost_cfg.top_exit_px,
    det_log_csv=_det_source,
    vial_rois=_vials,
    vial_expected_counts=vial_expected_counts,
    max_frames=None,
    watershed_cfg=dict(config.watershed),
)

save_run_params(OUTPUT_PATH, "tracker_output", {
    "ocsort_csv": OCSORT_CSV,
    "frames": int(df_wide.shape[0]),
    "track_count": int(df_wide.shape[1] - 1),
})

with open(os.path.join(OUTPUT_PATH, "tracker_log.json"), "w") as f:
    json.dump({
        "detection_log":      tracker.detection_log,
        "suppressed_tracks":  tracker.suppressed_tracks,
        "min_hits":           tracker.min_hits,
        "max_age":            tracker.max_age,
        "ghost_log":          getattr(tracker, "ghost_log",          []),
        "top_exit_events":    getattr(tracker, "top_exit_events",    []),
        "top_reentry_events": getattr(tracker, "top_reentry_events", []),
    }, f)

if write_overlays:
    render_detections_video(
        video_path=str(PATH_TO_VID),
        det_log_csv=_det_source,
        out_mp4=os.path.join(OUTPUT_PATH, f"{short_name}_detections_RF-DETR.mp4"),
    )

In [ ]:
# Mid-pipeline check: are detections actually reaching the tracker?
# No vial assignment yet, so no per-vial report saved.
run_diagnostics(
    tracker=tracker,
    df_wide=df_wide,
    n_expected=sum(vial_expected_counts.values()),
    fps=fps,
    config=config,
)

## 5. Vial assignment and ordered IDs

Melt the wide OC-SORT CSV to long format, then assign each detection to a vial via the ROI JSON. `ordered_id` is a left-to-right sequential index within each vial.

In [ ]:
vial_rois, _ = load_vial_rois(ROI_JSON)  # handles both flat and {bbox, n_flies} formats

long_df = wide_to_long(pd.read_csv(OCSORT_CSV), out_csv=LONG_CSV)
df_ordered = assign_vials_and_ordered_ids(
    ocsort_csv=LONG_CSV,
    roi_json=ROI_JSON,
    out_csv=ORDERED_CSV,
    fps=fps,
)

save_run_params(OUTPUT_PATH, "ordered", {
    "csv": ORDERED_CSV,
    "rows": int(df_ordered.shape[0]),
    "track_count": int(df_ordered["ordered_id"].nunique()),
})
df_ordered.head()

## 6. Diagnostics

Full diagnostics report (per-vial track counts vs expected, suppressed tracks, re-link impact, coverage histogram). Writes `metrics_report.md` and `metrics_report.html` into the run folder.

In [ ]:
run_diagnostics(
    tracker=tracker,
    df_wide=df_wide,
    df_ordered=df_ordered,
    n_expected=sum(vial_expected_counts.values()),
    fps=fps,
    vial_rois=vial_rois,
    config=config,
    output_dir=OUTPUT_PATH,
)

## 7. Overlay videos

Skipped when `visualization.enabled: false` in config.yaml. When enabled: raw OC-SORT IDs and ordered (within-vial) IDs. Substrate from `config.visualization.overlay_source` (`raw_cropped` preferred when preprocessing produced one).

In [ ]:
if write_overlays:
    _mode = config.visualization.overlay_source.lower()
    if _mode == "raw_cropped" and RAW_CROPPED_VIDEO is not None:
        OVERLAY_VIDEO = str(RAW_CROPPED_VIDEO)
    elif _mode == "raw_cropped":
        OVERLAY_VIDEO = str(RAW_VIDEO)
    else:
        OVERLAY_VIDEO = str(PATH_TO_VID)

    render_raw_overlay_video(
        video_path=OVERLAY_VIDEO,
        csv_path=LONG_CSV,
        out_mp4=RAW_OVERLAY_MP4,
        vial_rois=vial_rois,
        det_log_csv=DET_LOG_CSV,
    )
    render_vial_overlay_video(
        video_path=OVERLAY_VIDEO,
        csv_path=ORDERED_CSV,
        out_mp4=OVERLAY_MP4,
        vial_rois=vial_rois,
        det_log_csv=DET_LOG_CSV,
    )

    save_run_params(OUTPUT_PATH, "outputs", {"raw_overlay": RAW_OVERLAY_MP4, "ordered_overlay": OVERLAY_MP4})
    Video(RAW_OVERLAY_MP4, width=800)
else:
    print("Overlays skipped (visualization.enabled=false in config.yaml)")